In [1]:
from IPython.display import display 

from datetime import datetime

import pandas as pd 
import missingno as msno
import numpy as np
from scipy.stats import zscore

import shutil
import os

from datetime import timedelta

In [2]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from config import gam_info

import test_functions 
import functions 

In [3]:
platformID = 'CSS'

In [4]:
# country
pop_size_col = gam_info['population_column']

country_cols = ['PlaceID', pop_size_col]
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)[country_cols]
country_codes[pop_size_col] = ( pd.to_numeric(country_codes[pop_size_col], errors='coerce')
                                    .fillna(1)
                                    .astype(int))

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period',)
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])
today = pd.Timestamp.today()
last_monday = today - timedelta(days=today.weekday() + 7)
valid_weeks = week_tester[week_tester['w/c'] <= last_monday]
number_of_weeks = valid_weeks['WeekNumber_finYear'].max()

service_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='ServiceID',)

platform_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='PlatformID',)


### RUN Tests
test_functions.test_lookup_files(country_codes, country_cols, [f"{platformID}_0", f"{platformID}_1", f"{platformID}_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_3", f"{platformID}_4", f"{platformID}_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(service_tester, ['ServiceID'], [f"{platformID}_6", f"{platformID}_7", f"{platformID}_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")

test_functions.test_lookup_files(platform_tester, ['PlatformID'], [f"{platformID}_9", f"{platformID}_10", f"{platformID}_11"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test CSS_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test CSS_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test CSS_2 passed: No empty values in lookup.
...updating logbook...

✅ Test CSS_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test CSS_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test CSS_5 passed: No empty values in lookup.
...updating logbook...

✅ Test CSS_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test CSS_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test CSS_8 passed: No empty values in lookup.
...updating logbook...

✅ Test CSS_9 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test CSS_10 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test CSS_11 passed: No empty values in lookup.
...updating logbook...



In [5]:
# TODO Add tests (recognising all services, countries, platforms) to all the overlap sheets that are used here 
# overlap
overlap_nonHeavy = pd.read_excel("../data/stale/Final Overlaps 2021.xlsx", sheet_name='non heavy')
overlap_nonHeavy = overlap_nonHeavy.rename(columns={'Week': 'WeekNumber_finYear', 
                                                    'Service Code': 'ServiceID', 
                                                    'GeoCode': 'PlaceID'})
# ASK MINNIE WHY THE SHEET HAS MORE THAN ONE VALUE PER SERVICE/PLACE/WEEK
overlap_nonHeavy = overlap_nonHeavy.drop_duplicates(subset=['PlaceID', 'ServiceID', 'WeekNumber_finYear'], 
                                                    keep='first')
overlap_nonHeavy = overlap_nonHeavy.merge(week_tester[['w/c', 'WeekNumber_finYear']], 
                                          on='WeekNumber_finYear', 
                                          how='outer').drop(columns=['WeekNumber_finYear'])

overlap_nonHeavyAdd = pd.read_excel("../data/stale/Final Overlaps 2021.xlsx", sheet_name='non heav additional')
overlap_nonHeavyAdd = overlap_nonHeavyAdd.rename(columns={'GeoCode': 'PlaceID'})

overlap_SocWebOverlap = pd.read_excel("../data/stale/Final Overlaps 2021.xlsx", sheet_name='SocWebOverlap').drop_duplicates()
overlap_SocWebOverlap['PlaceID'] = overlap_SocWebOverlap['PlaceID'].replace('MYT', 'MAY').replace('WLF', 'WFI')
overlap_SocWebOverlap = overlap_SocWebOverlap.merge(country_codes, on='PlaceID', how='left')

overlap_referral = pd.read_excel("../data/stale/Final Overlaps 2021.xlsx", sheet_name='Referrals').drop_duplicates()
overlap_referral = overlap_referral.rename(columns={'Week Number': 'WeekNumber_finYear', 
                                                    'ServiceID': 'ServiceID', 
                                                    'Country Code': 'PlaceID',
                                                    '% Social': '%_AnalyticsSocialOverlap'})
overlap_referral = overlap_referral.merge(week_tester[['w/c', 'WeekNumber_finYear']], 
                                          on='WeekNumber_finYear', 
                                          how='outer').drop(columns=['WeekNumber_finYear'])

analytics_socialOverlap = 0.397690544

cols = ['PlaceID', 'digiGAM_FOA_WT-']
africa_dedup_countries = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID')[cols]


# calculate CSS

## import data 

### site

In [6]:
cols = ['YearGAE', 'w/c', 'ServiceID', 'PlatformID', 'PlaceID', 'Reach']
try:
    site_weekly_df = pd.read_csv(f"../data/singlePlatform/site/weekly/{gam_info['file_timeinfo']}_site_reach_weekly.csv")[cols]
    site_weekly_df['w/c'] = pd.to_datetime(site_weekly_df['w/c'])
    
    test_columns = {
    'PlaceID': country_codes['PlaceID'].tolist(),
    'w/c': week_tester['w/c'].tolist(),
    'ServiceID': service_tester['ServiceID'].tolist(),
    'PlatformID': platform_tester['PlatformID'].tolist()
    }
    
    for i, (column, allowed_values) in enumerate(test_columns.items(), start=1):
        label = f"total_digi_{i}"
        test_functions.test_allowed_values(site_weekly_df, column, allowed_values, label, 'site_ingest')
# test duplicates in sites
except:
    site_weekly_df = pd.DataFrame()
    print('site not there!')


✅ Pass - found only allowed values
...updating logbook...

✅ Pass - found only allowed values
...updating logbook...

✅ Pass - found only allowed values
...updating logbook...

✅ Pass - found only allowed values
...updating logbook...



### social

In [7]:
social_weekly_df = pd.read_csv("../data/combinePlatforms/GAM2026_weekly_WSC.csv")

social_weekly_df['YearGAE'] = gam_info['YearGAE']
social_weekly_df = social_weekly_df[['ServiceID', 'PlaceID', 'PlatformID', 'w/c', 'YearGAE', 'Reach']]


test_columns = {
    'PlaceID': country_codes['PlaceID'].tolist(),
    'w/c': week_tester['w/c'].tolist(),
    'ServiceID': service_tester['ServiceID'].tolist(),
    'PlatformID': platform_tester['PlatformID'].tolist()
}

for i, (column, allowed_values) in enumerate(test_columns.items(), start=5):
    print(column)
    label = f"total_digi_0{i}"
    test_functions.test_allowed_values(social_weekly_df, column, allowed_values, label, 'social_ingest')
social_weekly_df.head()

PlaceID
✅ Pass - found only allowed values
...updating logbook...

w/c
❌ Fail - found not allowed values
...updating logbook...

ServiceID
✅ Pass - found only allowed values
...updating logbook...

PlatformID
✅ Pass - found only allowed values
...updating logbook...



,ServiceID,PlaceID,PlatformID,w/c,YearGAE,Reach
0,AFA,AFG,WSC,2025-03-31,2026,87.206038
1,AMH,AFG,WSC,2025-03-31,2026,47.303793
2,ARA,AFG,WSC,2025-03-31,2026,5586.954972
3,AZE,AFG,WSC,2025-03-31,2026,371.988439
4,BEN,AFG,WSC,2025-03-31,2026,1194.896463


### combine site & social

In [8]:
site_social_df = pd.concat([site_weekly_df, social_weekly_df], ignore_index=True)
site_social_df['w/c'] = pd.to_datetime(site_social_df['w/c'])
site_social_df = site_social_df.pivot(index=['YearGAE', 'w/c', 'ServiceID', 'PlaceID'],
                                      columns="PlatformID", values="Reach").fillna(0).reset_index()
site_social_df.sample(5)

PlatformID,YearGAE,w/c,ServiceID,PlaceID,WDI,WIN,WSC,WWW
23436,2026,2025-04-14,GUJ,GHA,3.943658,0.000000,461.327011,3.905819
380240,2026,2025-12-15,RUS,MEX,855.853544,44.135302,397.635263,898.145774
363535,2026,2025-12-08,ANW,GUA,35623.287837,2421.396730,143732.956899,37654.367194
269704,2026,2025-09-29,URD,SAF,871.086464,6.759035,19837.735809,883.570192
363969,2026,2025-12-08,ARA,CAN,12200.312881,6502.785365,31022.292770,22751.067554


In [9]:
non_MA_WOR = site_social_df[~site_social_df['ServiceID'].isin(['WOR', 'MA-'])]

exclude_ids = ['WOR', 'MA-', 
               'ENG', 'EN2', 'ENW', 
               'ANW', 'TOT', 'AX2', 'ANY', 'ALL', ]
ws_site_social = site_social_df[~site_social_df['ServiceID'].isin(exclude_ids)]


ma_wor_df = site_social_df[site_social_df['ServiceID'].isin(['WOR', 'MA-'])]


## WSL

### determine & handle outlier

In [10]:

# Ensure required columns exist
for col in ['WSC', 'WDI']: 
    if col not in ws_site_social.columns:
        ws_site_social[col] = 0  # Fill missing column with zeros

# Group by 'service' and 'w/c', summing the numerical columns
grouped = ws_site_social.groupby(['ServiceID', 'w/c'], as_index=False)[['WSC', 'WDI']].sum()

# Compute Z-scores within each 'service' group
def compute_zscores(group):
    group['WSC_z'] = (group['WSC'] - group['WSC'].mean()) / group['WSC'].std(ddof=1)
    group['WDI_z'] = (group['WDI'] - group['WDI'].mean()) / group['WDI'].std(ddof=1)
    return group

outlier_df = grouped.groupby('ServiceID', group_keys=False).apply(compute_zscores)
outlier_df = outlier_df[(outlier_df['WSC_z'] > 1.96) | (outlier_df['WDI_z'] > 1.96)]
outlier_df = outlier_df[['w/c', 'ServiceID']]


/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_16752/2629149810.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  outlier_df = grouped.groupby('ServiceID', group_keys=False).apply(compute_zscores)


In [11]:
# 2. Identify outliers and non-outliers
merged_outlier = ws_site_social.merge(outlier_df, on=['w/c', 'ServiceID'], how='outer', 
                                   indicator=True)


In [12]:
outliers = merged_outlier[merged_outlier['_merge'] == 'both']
print(outliers.shape)

# 3. Check for overlap with non-heavy overlaps
outliers = outliers.drop(columns=['_merge'])
outliers = outliers.merge(overlap_nonHeavy.rename(columns={'Non Heavy - Month': 'NonHeavy1'}), 
                                 on=['ServiceID', 'PlaceID', 'w/c'], 
                                 how='left')
print(outliers.shape)

# 4. Add additional overlap info
outliers = outliers.merge(overlap_nonHeavyAdd.rename(columns={'Avg_%': 'NonHeavy2'}), 
                                        on='PlaceID', 
                                        how='left')
print(outliers.shape)

outliers['NonHeavy3'] = 0.593887
outliers['%_NonHeavy'] = np.where(outliers['NonHeavy1'].notna(), 
                                         outliers['NonHeavy1'],
                                         np.where(outliers['NonHeavy2'].notna(), 
                                                  outliers['NonHeavy2'], 
                                                  outliers['NonHeavy3']))
outliers = outliers.drop(columns=['NonHeavy1', 'NonHeavy2', 'NonHeavy3'])

outliers['NonHeavyWDI'] = outliers['WDI'] * outliers['%_NonHeavy']
outliers['HeavyWDI'] = outliers['WDI'] * (1-outliers['%_NonHeavy'])

outliers['NonHeavyWSC'] = outliers['WSC'] * outliers['%_NonHeavy']
outliers['HeavyWSC'] = outliers['WSC'] * (1-outliers['%_NonHeavy'])

# be aware - PCN is missing 
outliers = outliers.merge(overlap_SocWebOverlap, on='PlaceID', how='left')
print(outliers.shape)

# Apply to your DataFrame
outliers = functions.sainsbury_formula(outliers, 'Population 2020', ['NonHeavyWDI', 'NonHeavyWSC'], 'NonHeavy_WSC_WDI')

# BUFFER
def compute_heavy_combined(row):
    heavy_wdi = row['HeavyWDI']
    heavy_wsc = row['HeavyWSC']
    social_inc = row['Social Incremental']
    site_inc = row['Web']
    
    if heavy_wdi > heavy_wsc:
        return heavy_wdi + heavy_wsc * social_inc
    else:
        return heavy_wsc + heavy_wdi * site_inc

# Apply to your DataFrame
outliers['BUFFER_Heavy_WSC_WDI'] = outliers.apply(compute_heavy_combined, axis=1)
outliers['Buffer_WSC_WDI'] = outliers['NonHeavy_WSC_WDI'] + outliers['BUFFER_Heavy_WSC_WDI']
outliers['Heavy_WSC_WDI'] = (outliers['HeavyWSC'] + outliers['HeavyWDI']) * outliers['Own Web & Social Factor']
outliers['WSC_WDI'] = outliers['Heavy_WSC_WDI'] + outliers['NonHeavy_WSC_WDI']

def compute_wsc_wdi_buffer(row):
    wsc_wdi = row['WSC_WDI']
    wdi = row['WDI']
    wsc = row['WSC']
    buffer = row['Buffer_WSC_WDI']
    
    if wsc_wdi < wdi:
        return buffer
    elif wsc_wdi < wsc:
        return buffer
    else:
        return wsc_wdi

# Ensure required columns exist
for col in ['WIN', 'WWW']: 
    if col not in outliers.columns:
        outliers[col] = 0  # Fill missing column with zeros

outliers['WSC_WDI'] = outliers.apply(compute_wsc_wdi_buffer, axis=1)
outliers = outliers[['PlaceID', 'ServiceID', 'YearGAE', 'w/c', 
                     'WDI', 'WIN', 'WSC', 'WWW', 
                     'WSC_WDI']]

(27775, 9)
(27775, 9)
(27775, 10)
(27775, 30)


### handle non outlier

In [13]:
no_outliers = merged_outlier[merged_outlier['_merge'] == 'left_only']
overlap_cols = ['PlaceID', pop_size_col, 'Own Web & Social Factor', 'Web', 'Social Incremental']
no_outliers = no_outliers.merge(overlap_SocWebOverlap[overlap_cols], on='PlaceID', how='left')


def compute_web_social_value(row):
    wdi = row['WDI']
    wsc = row['WSC']
    factor = row['Own Web & Social Factor']
    
    if wsc == 0:
        return wdi
    elif wdi == 0:
        return wsc
    else:
        return (wdi + wsc) * factor

# Apply to your DataFrame
no_outliers['NewTapestryDigi'] = no_outliers.apply(compute_web_social_value, axis=1)

def compute_web_social_mix(row):
    wdi = row['WDI']
    wsc = row['WSC']
    web = row['Web']
    social_inc = row['Social Incremental']
    
    if wsc > wdi:
        return wsc + wdi * web
    else:
        return wdi + wsc * social_inc

# Apply to your DataFrame
no_outliers['BufferForTapestry'] = no_outliers.apply(compute_web_social_mix, axis=1)


def compute_tapestry_buffer(row):
    digi = row['NewTapestryDigi']
    wsc = row['WSC']
    wdi = row['WDI']
    buffer = row['BufferForTapestry']
    
    if digi < wsc:
        return buffer
    elif digi < wdi:
        return buffer
    else:
        return digi

# Apply to your DataFrame
# Ensure required columns exist
for col in ['WIN', 'WWW']: 
    if col not in no_outliers.columns:
        no_outliers[col] = 0  # Fill missing column with zeros

no_outliers['WSC_WDI'] = no_outliers.apply(compute_tapestry_buffer, axis=1)
no_outliers = no_outliers[['PlaceID', 'ServiceID', 'YearGAE', 'w/c',
                           'WDI', 'WIN', 'WSC', 'WWW',
                           'WSC_WDI']]

### calculate CSS

In [14]:
ws_site_social_postOutlier = pd.concat([outliers, no_outliers])
ws_site_social_postOutlier['%_OverlapWithOwnSite'] = ((ws_site_social_postOutlier['WSC'] +
                                                     ws_site_social_postOutlier['WDI']) -
                                                    ws_site_social_postOutlier['WSC_WDI'])/ws_site_social_postOutlier['WDI']

ws_site_social_postOutlier = ws_site_social_postOutlier.merge(overlap_referral, 
                                                              on=['PlaceID', 'ServiceID', 'w/c'],
                                                              how='left', indicator=True)

ws_site_social_postOutlier['%_AnalyticsSocialOverlap'] = ws_site_social_postOutlier['%_AnalyticsSocialOverlap'].fillna(analytics_socialOverlap)


def compute_site_overlap_adjustment(row):
    site_overlap = row['%_OverlapWithOwnSite']
    analytics_overlap = row['%_AnalyticsSocialOverlap']
    wdi = row['WDI']
    wsc = row['WSC']
    wsc_wdi = row['WSC_WDI']
    
    if site_overlap < analytics_overlap:
        return (wdi + wsc) - (wdi * analytics_overlap)
    else:
        return wsc_wdi
ws_site_social_postOutlier['Pegged_WSC_WDI'] = ws_site_social_postOutlier.apply(compute_site_overlap_adjustment, axis=1)

def flag_pegged_wsc_wdi(row):
    pegged = row['Pegged_WSC_WDI']
    wsc = row['WSC']
    wdi = row['WDI']
    
    if pegged < wsc:
        return "FLAG"
    elif pegged < wdi:
        return "FLAG"
    else:
        return "FINE"
ws_site_social_postOutlier['Check1'] = ws_site_social_postOutlier.apply(flag_pegged_wsc_wdi, axis=1)
print(ws_site_social_postOutlier['Check1'].value_counts())

def resolve_check_flag(row):
    check = row['Check1']
    wsc_wdi = row['WSC_WDI']
    pegged = row['Pegged_WSC_WDI']
    
    if check == "FLAG":
        return wsc_wdi
    else:
        return pegged

# Apply to your DataFrame
ws_site_social_postOutlier['final_WSC_WDI'] = ws_site_social_postOutlier.apply(resolve_check_flag, axis=1)

def compute_incremental_partner_reach(row):
    www = row['WWW']
    win = row['WIN']
    wdi = row['WDI']
    final_wsc_wdi = row['final_WSC_WDI']
    
    if www > win:
        return (www - wdi) + final_wsc_wdi
    elif win == 0:
        return final_wsc_wdi
    elif final_wsc_wdi == 0:
        return win
    else:
        return (www - wdi) + final_wsc_wdi

# Apply to your DataFrame
ws_site_social_postOutlier['Reach'] = ws_site_social_postOutlier.apply(compute_incremental_partner_reach, axis=1)

Check1
FINE    169155
FLAG    160712
Name: count, dtype: int64


In [15]:
cols = ['YearGAE', 'w/c', 'ServiceID', 'PlaceID', 'Reach']
weekly_ws_df =  ws_site_social_postOutlier[cols]

### annual average

In [16]:
annual_ws_df = weekly_ws_df.groupby(['YearGAE', 'ServiceID', 'PlaceID'])['Reach'].sum().reset_index()
annual_ws_df['Reach'] = annual_ws_df['Reach'] / number_of_weeks
annual_ws_df.head()

,YearGAE,ServiceID,PlaceID,Reach
0,2026,AFA,AFG,218.851407
1,2026,AFA,AGU,0.023319
2,2026,AFA,ALA,0.045801
3,2026,AFA,ALB,11.805962
4,2026,AFA,ALG,93.715193


## MA & Studios

In [17]:
ma_wor_df = site_social_df[site_social_df['ServiceID'].isin(['WOR', 'MA-'])]
ma_wor_df = ma_wor_df.merge(overlap_SocWebOverlap, on='PlaceID' , how='left')

# Ensure required columns exist
for col in ['WSC', 'WWW']: 
    if col not in ma_wor_df.columns:
        ma_wor_df[col] = 0  # Fill missing column with zeros

ma_wor_df = functions.sainsbury_formula(ma_wor_df, pop_size_col, ['WSC', 'WWW'], 'Reach')

cols = ['YearGAE', 'w/c', 'ServiceID', 'PlaceID', 'Reach']
weekly_ma_wor_df = ma_wor_df[cols]

annual_ma_wor_df = ma_wor_df.groupby(['YearGAE', 'ServiceID', 'PlaceID'])['Reach'].sum().reset_index()
annual_ma_wor_df['Reach'] = annual_ma_wor_df['Reach'] / number_of_weeks


## aggregated services

In [18]:
weekly_df = pd.concat([weekly_ws_df, weekly_ma_wor_df])
#weekly_df = weekly_df.merge(country_codes, on='PlaceID', how='left')
weekly_df.head()

,YearGAE,w/c,ServiceID,PlaceID,Reach
0,2026,2025-03-31,BUR,AFG,5852.089335
1,2026,2025-03-31,BUR,ALB,7.607994
2,2026,2025-03-31,BUR,ALG,1161.557382
3,2026,2025-03-31,BUR,AND,0.972487
4,2026,2025-03-31,BUR,ANG,18.477246


In [19]:
def compute_combined_reach(df, services, label, pop_size_col, country_codes, deal_with_zero=True, 
                          calc_type='sainsbury'):
    """
    Filters, merges, aggregates, and applies the Sainsbury formula to compute combined reach.

    Parameters:
    df (pd.DataFrame): Source DataFrame with weekly reach data.
    services (list): List of ServiceIDs to include.
    label (str): Label to assign to the resulting ServiceID.
    pop_size_col (str): Column name for population size.
    country_codes (pd.DataFrame): Mapping DataFrame for PlaceID enrichment.
    deal_with_zero (bool): Whether to apply shortcut logic in the formula.

    Returns:
    pd.DataFrame: Aggregated and transformed DataFrame with combined reach.
    """
    filtered_df = df[df['ServiceID'].isin(services)].merge(country_codes, on='PlaceID', how='left')
    
    pivot_df = pd.crosstab(
        index=[filtered_df['PlaceID'], filtered_df[pop_size_col], 
               filtered_df['w/c'], filtered_df['YearGAE']],
        columns=filtered_df['ServiceID'],
        values=filtered_df['Reach'],
        aggfunc='sum'
    ).reset_index().fillna(0)

    if calc_type == 'add':
        print("... using summation")
        pivot_df['Reach'] = pivot_df[services].sum(axis=1)
    elif calc_type == 'sainsbury':
        print("... using sainsbury")
        pivot_df = functions.sainsbury_formula(pivot_df, pop_size_col, services, 
                                               'Reach', deal_with_zero=deal_with_zero)
        
    else: 
        print('error')
        
    pivot_df['ServiceID'] = label
    return pivot_df[['YearGAE', 'w/c', 'ServiceID', 'PlaceID', 'Reach']]

### ENW

In [20]:
# Usage
enw_services = ['FOA', 'WSE']
enw_df = compute_combined_reach(weekly_df, enw_services, 'ENW', pop_size_col, country_codes)

... using sainsbury


### ENG

In [21]:
# Usage
eng_services = ['GNL', 'WSE']
eng_df = compute_combined_reach(weekly_df, eng_services, 'ENG', pop_size_col, country_codes)

... using sainsbury


### EN2

In [22]:

en2_services = ['ENG', 'WOR']
en2_df = compute_combined_reach(pd.concat([weekly_df, eng_df]), en2_services, 'EN2', pop_size_col, country_codes)


... using sainsbury


### AX2

In [23]:
ax2_services = [
    'AFA','AMH','ARA','AZE','BEN','BUR','DAR',#'ECH',
    'ELT','PER','FRE','GUJ','HAU','HIN','IGB','INO',
    'KOR','KRW','KYR','MAN','MAR','NEP','PAS','PDG','POR', 'POL', 'PUN','RUS','SER','SIN','SOM','SPA','SWA',
    'TAM','TEL','THA','TIG','TUR','UKR','URD','UZB','VIE','YOR', 'FOA', #'UKPS'
]

ax2_df = weekly_df[weekly_df['ServiceID'].isin(ax2_services)].merge(country_codes, on='PlaceID', how='left')
ax2_df = pd.crosstab(
                                        index = [ ax2_df['PlaceID'], 
                                                  ax2_df[pop_size_col], 
                                                  ax2_df['w/c'],
                                                  ax2_df['YearGAE']],
                                        columns = ax2_df['ServiceID'],
                                        values =  ax2_df['Reach'],
                                        aggfunc='sum'
                                    ).reset_index()
ax2_df = ax2_df.fillna(0)

temp2 = ax2_df.merge(africa_dedup_countries, on='PlaceID', how='left')
africa_df = temp2[~temp2['digiGAM_FOA_WT-'].isna()]
nonAfrica_df = temp2[temp2['digiGAM_FOA_WT-'].isna()]

# Apply the logic row-wise
def compute_value(row):
    others_sum = sum(row.get(code, 0) for code in ax2_services)
    if row['FOA'] > others_sum:
        return row['FOA'] + 0.60745497 * others_sum
    else:
        return others_sum + row['FOA'] * 0.60745497

africa_df['Reach'] = africa_df.apply(compute_value, axis=1)
nonAfrica_df = functions.sainsbury_formula(nonAfrica_df, gam_info['population_column'], ax2_services, 'Reach')
ax2_df = pd.concat([africa_df, nonAfrica_df])
ax2_df['ServiceID'] = 'AX2'
ax2_df = ax2_df[['YearGAE', 'w/c', 'ServiceID', 'PlaceID', 'Reach']]

ax2_df.head()

/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_16752/1906791123.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  africa_df['Reach'] = africa_df.apply(compute_value, axis=1)
/Users/dominiquebruns/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:610: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col_name] = df.apply(calculate_formula, axis=1)


,YearGAE,w/c,ServiceID,PlaceID,Reach
168,2026,2025-03-31,AX2,ALG,1.411735e+06
169,2026,2025-04-07,AX2,ALG,1.891763e+06
170,2026,2025-04-14,AX2,ALG,1.245553e+06
171,2026,2025-04-21,AX2,ALG,1.491441e+06
172,2026,2025-04-28,AX2,ALG,1.190262e+06


### ANW

In [24]:

anw_services = ['AX2', 'WSE']
anw_df = compute_combined_reach(pd.concat([weekly_df, ax2_df]), anw_services, 'ANW', 
                                pop_size_col, country_codes)


... using sainsbury


### ANY

In [25]:

any_services = ['ANW', 'GNL']
any_df = compute_combined_reach(pd.concat([weekly_df, anw_df]), any_services, 'ANY', 
                                pop_size_col, country_codes)


... using sainsbury


### TOT

In [26]:

tot_services = ['ANY', 'MA-']
tot_df = compute_combined_reach(pd.concat([weekly_df, any_df]), tot_services, 'TOT', 
                                pop_size_col, country_codes, calc_type='add')

... using summation


### ALL

In [27]:

all_services = ['TOT', 'WOR']
all_df = compute_combined_reach(pd.concat([weekly_df, tot_df]), all_services, 'ALL', 
                                pop_size_col, country_codes, calc_type='add')

... using summation


## finalising 

In [28]:
final_weekly_df = pd.concat([weekly_ws_df, weekly_ma_wor_df, enw_df, eng_df, en2_df, ax2_df, anw_df,
                     any_df, tot_df, all_df])

final_weekly_df['PlatformID'] = 'CSS'

final_weekly_df.to_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_weekly_CSS.csv", 
                       index=None)

In [29]:
final_annual_df = final_weekly_df.groupby(['YearGAE', 'ServiceID', 'PlatformID', 'PlaceID'])['Reach'].sum().reset_index()
final_annual_df['Reach'] = final_annual_df['Reach'] / number_of_weeks

final_annual_df.to_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_annual_CSS.csv", 
                       index=None)

# WT-

In [30]:
cols = ['YearGAE', 'w/c', 'ServiceID', 'PlatformID', 'PlaceID', 'Reach']

In [31]:
css_df = final_weekly_df.copy()[cols]
css_df['w/c'] = css_df['w/c'].dt.strftime('%Y-%m-%d')

In [32]:
podcast_df = pd.read_excel(f"../data/singlePlatform/podcast/weekly/{gam_info['file_timeinfo']}_podcast_data_weekly.xlsx")
podcast_df['YearGAE'] = gam_info['YearGAE']
podcast_df['w/c'] = podcast_df['w/c'].dt.strftime('%Y-%m-%d')
podcast_df = podcast_df[cols]
pod_df=podcast_df[podcast_df['PlatformID'].isin(['POD'])]
pod_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
235,2026,2025-03-31,ALL,POD,AFG,1080
236,2026,2025-03-31,ALL,POD,AGU,66
237,2026,2025-03-31,ALL,POD,ALA,89
238,2026,2025-03-31,ALL,POD,ALB,1588
239,2026,2025-03-31,ALL,POD,ALG,12088


In [33]:
wt_pod_df = pd.concat([css_df[cols], pod_df[cols]])

wt_pod_df=pd.DataFrame(wt_pod_df.pivot_table(index=['ServiceID','PlaceID', 'w/c', 'YearGAE'],
                                     columns='PlatformID', 
                                     values='Reach',
                                     aggfunc='sum')).reset_index()
wt_pod_df=wt_pod_df.merge(country_codes[['PlaceID', gam_info['population_column']]], 
                          on='PlaceID', how='left')
wt_pod_df=wt_pod_df.fillna(0)
wt_pod_df.head()

,ServiceID,PlaceID,w/c,YearGAE,CSS,POD,Population2020
0,AFA,AFG,2025-03-31,2026,96.152662,0.0,9237489
1,AFA,AFG,2025-04-07,2026,89.041538,0.0,9237489
2,AFA,AFG,2025-04-14,2026,84.923601,0.0,9237489
3,AFA,AFG,2025-04-21,2026,98.432661,0.0,9237489
4,AFA,AFG,2025-04-28,2026,116.265485,0.0,9237489


In [34]:
wt_pod_df = functions.sainsbury_formula(wt_pod_df, gam_info['population_column'],
                                                 ['CSS', 'POD'], 'Reach')
wt_pod_df['PlatformID'] = 'WT-'
wt_pod_df[(wt_pod_df['POD'] > 1) & (wt_pod_df['CSS'] > 1)].head()

,ServiceID,PlaceID,w/c,YearGAE,CSS,POD,Population2020,Reach,PlatformID
5974,ALL,AFG,2025-03-31,2026,1.377639e+06,1080.0,9237489,1.378558e+06,WT-
5975,ALL,AFG,2025-04-07,2026,1.448687e+06,1380.0,9237489,1.449851e+06,WT-
5976,ALL,AFG,2025-04-14,2026,1.081296e+06,1254.0,9237489,1.082404e+06,WT-
5977,ALL,AFG,2025-04-21,2026,1.194696e+06,1369.0,9237489,1.195888e+06,WT-
5978,ALL,AFG,2025-04-28,2026,1.435302e+06,1353.0,9237489,1.436445e+06,WT-


In [35]:
wt_pod_df[cols].to_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_weekly_WT-.csv", 
                       index=None)
wt_pod_df_annual = wt_pod_df.groupby(['YearGAE', 'ServiceID', 'PlatformID', 'PlaceID'])['Reach'].sum().reset_index()
wt_pod_df_annual['Reach'] = wt_pod_df_annual['Reach'] / number_of_weeks

wt_pod_df_annual.to_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_annual_CSS.csv", 
                       index=None)

# combine platforms

## import data

In [36]:
cols = ['YearGAE', 'w/c', 'ServiceID', 'PlatformID', 'PlaceID', 'Reach']

In [37]:
podcast_df = pd.read_excel(f"../data/singlePlatform/podcast/weekly/{gam_info['file_timeinfo']}_podcast_data_weekly.xlsx")
podcast_df['YearGAE'] = gam_info['YearGAE']
podcast_df['w/c'] = podcast_df['w/c'].dt.strftime('%Y-%m-%d')
podcast_df = podcast_df[cols]


In [38]:
try:
    site_df = pd.read_csv(f"../data/singlePlatform/site/weekly/{gam_info['file_timeinfo']}_site_reach_weekly.csv", index_col=0)
    site_df = site_df[cols]
    site_df['w/c'] = site_df['w/c'].dt.strftime('%Y-%m-%d')
    site_df.head()
except:
    print('site not there! ')
    site_df = pd.DataFrame()

site not there! 


In [39]:
# created from platform in 6.
social_wsc_df = pd.read_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_weekly_WSC.csv")
social_wsc_df = social_wsc_df.rename(columns={
    'Country Code': 'PlaceID',
    'Platform Code': 'PlatformID',
    'Service Code': 'ServiceID'
})
social_wsc_df['w/c'] = pd.to_datetime(social_wsc_df['w/c']).dt.strftime('%Y-%m-%d')
social_wsc_df['YearGAE'] = gam_info['YearGAE']
social_wsc_df = social_wsc_df[cols]
social_wsc_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
0,2026,2025-03-31,AFA,WSC,AFG,87.206038
1,2026,2025-03-31,AMH,WSC,AFG,47.303793
2,2026,2025-03-31,ARA,WSC,AFG,5586.954972
3,2026,2025-03-31,AZE,WSC,AFG,371.988439
4,2026,2025-03-31,BEN,WSC,AFG,1194.896463


In [40]:
# created from dataset per platform file in 5.
social_platforms_df = pd.read_csv(f"../data/combinePlatforms/social_media_data_{gam_info['file_timeinfo']}_platform_weekly.csv")
social_platforms_df['YearGAE'] = gam_info['YearGAE']
social_platforms_df['w/c'] = pd.to_datetime(social_platforms_df['w/c']).dt.strftime('%Y-%m-%d')
social_platforms_df = social_platforms_df[cols]
social_platforms_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
0,2026,2025-07-07,ENG,TTK,AFG,38935.180845
1,2026,2025-09-01,ENG,TTK,AFG,11154.424494
2,2026,2025-12-08,ENG,TTK,AFG,18288.030688
3,2026,2026-01-12,ENG,TTK,AFG,60569.635932
4,2026,2025-07-28,ENG,TTK,ALB,28892.635299


In [41]:
css_df = pd.read_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_weekly_CSS.csv")[cols]
css_df['w/c'] = pd.to_datetime(css_df['w/c']).dt.strftime('%Y-%m-%d')
css_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
0,2026,2025-03-31,BUR,CSS,AFG,5852.089335
1,2026,2025-03-31,BUR,CSS,ALB,7.607994
2,2026,2025-03-31,BUR,CSS,ALG,1161.557382
3,2026,2025-03-31,BUR,CSS,AND,0.972487
4,2026,2025-03-31,BUR,CSS,ANG,18.477246


In [42]:
wt_df = pd.read_csv(f"../data/combinePlatforms/{gam_info['file_timeinfo']}_weekly_WT-.csv")[cols]
wt_df['w/c'] = pd.to_datetime(wt_df['w/c']).dt.strftime('%Y-%m-%d')
wt_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
0,2026,2025-03-31,AFA,WT-,AFG,96.152662
1,2026,2025-04-07,AFA,WT-,AFG,89.041538
2,2026,2025-04-14,AFA,WT-,AFG,84.923601
3,2026,2025-04-21,AFA,WT-,AFG,98.432661
4,2026,2025-04-28,AFA,WT-,AFG,116.265485


# combine 

In [43]:
sources = {'pod': podcast_df, 
           'site': site_weekly_df, 
           'platform': social_platforms_df, 
           'wsc': social_wsc_df,
           'wt-': wt_df,
           'css': css_df}

def far_per_test(df):
    temp = df[df['ServiceID'].isin(['PER', 'FAR'])]
    display(temp.ServiceID.value_counts())
    if len(temp) > 0:
        df['ServiceID'] = df['ServiceID'].replace('FAR', 'PER')
    temp = df[df['ServiceID'].isin(['PER', 'FAR'])]
    display(temp.ServiceID.value_counts())
    return df

for name, source in sources.items():
    if len(source) > 0:
        print(f"\n{name}")
        sources[name] = far_per_test(source)
    else:
        print(f"\n{name} is empty! ")
digital_df = pd.concat(sources.values())

# Paths for final outputs
output_dir = "../data/final"
os.makedirs(output_dir, exist_ok=True)

digital_df['w/c'] = pd.to_datetime(digital_df['w/c']).dt.strftime('%Y-%m-%d')
digital_df = digital_df[digital_df['ServiceID'] != 'AXE']


pod


ServiceID
PER    9044
Name: count, dtype: int64

ServiceID
PER    9044
Name: count, dtype: int64


site


ServiceID
PER    17167
Name: count, dtype: int64

...

ServiceID
PER    8308
Name: count, dtype: int64

ServiceID
PER    8308
Name: count, dtype: int64


css


ServiceID
PER    8261
Name: count, dtype: int64

ServiceID
PER    8261
Name: count, dtype: int64

In [44]:
digital_df[digital_df['w/c'] == '2026-01-05']['PlatformID'].unique()

array(['DPO', 'POD', 'WDI', 'WWW', 'WIN', 'TTK', 'FBE', 'INS', 'TWI',
       'YT-', 'WSC', 'WT-', 'CSS'], dtype=object)

In [45]:

test_combi = digital_df[digital_df['Reach'].fillna(0) > 0]

test_combi = test_combi[['PlatformID', 'ServiceID']].drop_duplicates()
test_combi['Start'] = digital_df['w/c'].min()
test_combi['End'] = digital_df['w/c'].max()
test_combi.head()

,PlatformID,ServiceID,Start,End
0,DPO,ALL,2025-03-31,2026-01-12
235,POD,ALL,2025-03-31,2026-01-12
470,DPO,ANW,2025-03-31,2026-01-12
704,POD,ANW,2025-03-31,2026-01-12
938,DPO,ARA,2025-03-31,2026-01-12


In [46]:
digital_df.head()

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
0,2026,2025-03-31,ALL,DPO,AFG,1080.0
1,2026,2025-03-31,ALL,DPO,AGU,66.0
2,2026,2025-03-31,ALL,DPO,ALA,89.0
3,2026,2025-03-31,ALL,DPO,ALB,1588.0
4,2026,2025-03-31,ALL,DPO,ALG,12088.0


In [47]:
# missing platforms 

# missing services 

# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['ServiceID', 'PlatformID'],
                                               main_data=digital_df,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=test_combi,
                                               test_number=f"{platformID}_8_07",
                                               test_step="Check all weeks present for each account")


/Users/dominiquebruns/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:110: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  main_test_data[key] = pd.to_datetime(main_test_data[key], errors='coerce', dayfirst=True)


❌ Missing weeks from Start onward:
      ServiceID PlatformID       Start         End        w/c
14910       AFA        YT-  2025-03-31  2026-01-12 2025-03-31
7266        DAR        WDI  2025-03-31  2026-01-12 2025-03-31
7308        DAR        WWW  2025-03-31  2026-01-12 2025-03-31
7686        EN2        WIN  2025-03-31  2026-01-12 2025-03-31
7560        ENG        WIN  2025-03-31  2026-01-12 2025-03-31
...         ...        ...         ...         ...        ...
7726        EN2        WIN  2025-03-31  2026-01-12 2026-01-05
7475        ENW        WIN  2025-03-31  2026-01-12 2026-01-12
7727        EN2        WIN  2025-03-31  2026-01-12 2026-01-12
7601        ENG        WIN  2025-03-31  2026-01-12 2026-01-12
7139        WSE        WIN  2025-03-31  2026-01-12 2026-01-12

[421 rows x 5 columns]

Summary of missing weeks per channel_id_col:
   ServiceID PlatformID  missing_week_count
6        DAR        WWW                  31
5        DAR        WDI                  31
8        ENG       

,ServiceID,PlatformID,Start,End,w/c
14910,AFA,YT-,2025-03-31,2026-01-12,2025-03-31
7266,DAR,WDI,2025-03-31,2026-01-12,2025-03-31
7308,DAR,WWW,2025-03-31,2026-01-12,2025-03-31
7686,EN2,WIN,2025-03-31,2026-01-12,2025-03-31
7560,ENG,WIN,2025-03-31,2026-01-12,2025-03-31
...,...,...,...,...,...
7726,EN2,WIN,2025-03-31,2026-01-12,2026-01-05
7475,ENW,WIN,2025-03-31,2026-01-12,2026-01-12
7727,EN2,WIN,2025-03-31,2026-01-12,2026-01-12
7601,ENG,WIN,2025-03-31,2026-01-12,2026-01-12


In [48]:
week_tester['w/c'].max()

Timestamp('2026-03-23 00:00:00')

In [49]:
digital_df[(digital_df['ServiceID'] == 'SPA') & 
            (digital_df['PlatformID'] == 'TTK') & 
            (digital_df['w/c'] == '2026-01-05')]

,YearGAE,w/c,ServiceID,PlatformID,PlaceID,Reach
25330,2026,2026-01-05,SPA,TTK,ARG,5.219368e+05
26068,2026,2026-01-05,SPA,TTK,BOL,8.491678e+04
26876,2026,2026-01-05,SPA,TTK,CHL,4.797581e+05
27055,2026,2026-01-05,SPA,TTK,COL,1.065375e+06
27122,2026,2026-01-05,SPA,TTK,COS,2.906216e+04
27298,2026,2026-01-05,SPA,TTK,ECU,5.469875e+05
27411,2026,2026-01-05,SPA,TTK,ELS,1.622457e+03
28542,2026,2026-01-05,SPA,TTK,GUA,2.264364e+05
30421,2026,2026-01-05,SPA,TTK,MEX,1.673188e+06
30939,2026,2026-01-05,SPA,TTK,NIC,3.054389e+04


In [50]:
digital_df.to_csv(f"{output_dir}/{gam_info['file_timeinfo']}_digi_gam_weekly.csv", 
                       index=None)
digital_df.to_csv(f"../../../New Lumen 2025/Datasets/Raw/{gam_info['file_timeinfo']}_digi_gam_weekly.csv", 
                       index=None)

digital_annual_df = digital_df.groupby(['YearGAE', 'ServiceID', 'PlatformID', 'PlaceID'])['Reach'].sum().reset_index()
digital_annual_df['Reach'] = digital_annual_df['Reach'] / number_of_weeks

digital_annual_df.to_csv(f"{output_dir}/{gam_info['file_timeinfo']}_digi_gam_annual.csv", 
                       index=None)
digital_annual_df.to_csv(f"../../../New Lumen 2025/Datasets/Raw/{gam_info['file_timeinfo']}_digi_gam_annual.csv", 
                       index=None)

# ✅ Copy the test logbook into the same folder
today_date = datetime.now().strftime('%Y-%m-%d')
file_path = f"../test/issue_lists_{today_date}"
logbook_src = f"{file_path}/test_logbook.xlsx"
logbook_dest = f"{output_dir}/{gam_info['file_timeinfo']}_test_logbook.xlsx"

if os.path.exists(logbook_src):
    shutil.copy(logbook_src, logbook_dest)
    print(f"Test logbook copied to: {logbook_dest}")
else:
    print("Warning: Test logbook not found!")

In [51]:
# add a test - for all platform services and weeks is there a gap? 

In [54]:
digital_df[digital_df['w/c'] == '2026-01-12']['PlatformID'].unique()

array(['DPO', 'POD', 'WDI', 'WWW', 'WIN', 'TTK', 'FBE', 'INS', 'TWI',
       'YT-', 'WSC', 'WT-', 'CSS'], dtype=object)

In [55]:
digital_df[digital_df['w/c'] == '2026-01-12']['ServiceID'].unique()

array(['ALL', 'ANW', 'ARA', 'AX2', 'AZE', 'BUR', 'ELT', 'HAU', 'HIN',
       'INO', 'KRW', 'KYR', 'MAN', 'MAR', 'NEP', 'PER', 'POR', 'RUS',
       'TAM', 'UKPS', 'UKR', 'WSE', 'TOT', 'ANY', 'ENG', 'EN2', 'AFA',
       'AMH', 'BEN', 'DAR', 'ECH', 'FRE', 'GNL', 'GUJ', 'IGB', 'KOR',
       'PAS', 'PDG', 'POL', 'PUN', 'SER', 'SIN', 'SOM', 'SPA', 'SWA',
       'TEL', 'THA', 'TIG', 'TUR', 'URD', 'UZB', 'VIE', 'WOR', 'YOR',
       'ENW', 'BNO', 'FOA', 'MA-', 'BNI'], dtype=object)